# 📘 데코레이터

**데코레이터**는 함수의 동작을 수정하거나 확장하는 함수입니다.
`@decorator` 문법으로 간결하게 적용합니다.

**학습 목표:**
- 데코레이터 기본 개념과 작성법
- 매개변수가 있는 데코레이터
- 여러 데코레이터 겹치기
- functools.wraps로 함수 정보 보존

## 1. 데코레이터 기본

데코레이터는 **함수를 인수로 받아 새로운 함수를 반환**하는 함수입니다.
`@decorator`는 `func = decorator(func)`의 간략한 표현입니다.

In [ ]:
# ┌───────────────────────────────────────────────┐
# │  데코레이터 구조                                  │
# │  def my_decorator(func):    ← 함수를 인수로 받음    │
# │      def wrapper(*args, **kwargs):                │
# │          # 함수 호출 전 실행할 코드                │
# │          result = func(*args, **kwargs)            │
# │          # 함수 호출 후 실행할 코드                │
# │          return result                             │
# │      return wrapper                                │
# │                                                    │
# │  @my_decorator                                     │
# │  def my_function():                                │
# │      ...                                           │
# │  # my_function = my_decorator(my_function)         │
# └───────────────────────────────────────────────┘


In [ ]:
# 기본 데코레이터
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print(f"Before calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"After calling {func.__name__}")
        return result
    return wrapper

@my_decorator
def say_hello(name):
    print(f"Hello, {name}!")

say_hello("Alice")


In [ ]:
# Before calling say_hello
# Hello, Alice!
# After calling say_hello


In [ ]:
# 반환값을 수정하는 데코레이터
def double_result(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result * 2
    return wrapper

@double_result
def add(a, b):
    return a + b

print(f"add(3, 5) = {add(3, 5)}")   # 16 (=(3+5)*2)


## 2. 실용적인 데코레이터 예시

실행 시간 측정, 로깅, 권한 검사 등에 데코레이터를 활용합니다.

In [ ]:
import time
from functools import wraps


In [ ]:
# ┌─────────────────────────────────────────┐
# │  functools.wraps                         │
# │  데코레이터가 원래 함수의 이름과            │
# │  docstring을 보존하도록 합니다             │
# │  모든 데코레이터에 사용 권장!              │
# └─────────────────────────────────────────┘


In [ ]:
# 실행 시간 측정 데코레이터
def timer(func):
    @wraps(func)    # 원래 함수 정보 보존
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"{func.__name__} 실행 시간: {end - start:.4f}초")
        return result
    return wrapper

@timer
def slow_sum(n):
    """1부터 n까지의 합을 구합니다"""
    return sum(range(n + 1))

result = slow_sum(1_000_000)
print(f"결과: {result}")
print(f"함수 이름: {slow_sum.__name__}")    # slow_sum (wraps 덕분)
print(f"docstring: {slow_sum.__doc__}")     # 보존됨


In [ ]:
# ┌─────────────────────────────────────────┐
# │  여러 데코레이터 겹치기                    │
# │  아래에서 위로 적용됩니다                   │
# │  @bold → @italic → greet 순서로           │
# │  greet = bold(italic(greet))              │
# └─────────────────────────────────────────┘

def bold(func):


In [ ]:
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"**{func(*args, **kwargs)}**"
    return wrapper

def italic(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"*{func(*args, **kwargs)}*"
    return wrapper

@bold
@italic
def greet(name):
    return f"Hello, {name}"

print(greet("Alice"))   # ** *Hello, Alice* ** (bold가 italic을 감쌈)


## 3. 매개변수가 있는 데코레이터

데코레이터 자체에 인수를 전달하려면 **3중 중첩 함수**가 필요합니다.

In [ ]:
# ┌───────────────────────────────────────────────┐
# │  매개변수가 있는 데코레이터                        │
# │  def decorator_with_args(deco_args):             │
# │      def decorator(func):                        │
# │          def wrapper(*args, **kwargs):            │
# │              # deco_args 사용 가능                │
# │              return func(*args, **kwargs)          │
# │          return wrapper                            │
# │      return decorator                              │


In [ ]:
# └───────────────────────────────────────────────┘

# n번 반복 실행하는 데코레이터
def repeat(n):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            results = []
            for _ in range(n):
                results.append(func(*args, **kwargs))
            return results
        return wrapper
    return decorator

@repeat(3)
def greet(name):
    return f"Hello, {name}!"

print(greet("Alice"))


In [ ]:
# ['Hello, Alice!', 'Hello, Alice!', 'Hello, Alice!']

# 권한 검사 데코레이터
def require_role(role):
    def decorator(func):
        @wraps(func)
        def wrapper(user_role, *args, **kwargs):
            if user_role != role:
                return f"접근 거부: {role} 권한 필요"
            return func(user_role, *args, **kwargs)
        return wrapper
    return decorator

@require_role("admin")
def delete_database(role):
    return "데이터베이스 삭제 완료"

print(delete_database("admin"))    # 데이터베이스 삭제 완료
print(delete_database("user"))     # 접근 거부: admin 권한 필요


## 🎯 연습 문제

1. 함수의 실행 결과를 `"결과: {결과}"` 형태로 감싸는 `format_result` 데코레이터를 작성하세요.
2. 함수의 인수가 양수인지 확인하는 `validate_positive` 데코레이터를 작성하세요. (음수면 "양수만 가능합니다" 반환)
3. `@repeat(2)` 데코레이터를 사용해 함수를 2번 실행하고 결과를 리스트로 반환하세요.
4. `functools.wraps`를 적용해 데코레이터가 원래 함수의 `__name__`과 `__doc__`을 보존하는지 확인하세요.

In [ ]:
# 연습 문제 풀이
